In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "database").is_dir() and (p / "code").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abra o notebook em Codigos ou em uma de suas subpastas.")
DATABASE_DIR = PROJECT_ROOT / "database"
RESULT_DIR = PROJECT_ROOT / "result" / "treinamento_estatico"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
# Cada experimento mantém seus próprios dataframes derivados.
PROCESSED_DIR = RESULT_DIR / "processados"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / "code" / "visualizacao"))


# Fake.Br — treinamento com divisão temporal

Notebook iniciado em 05/08/2026.

Neste experimento:

- **Treino:** notícias publicadas até **31/12/2019**, inclusive.
- **Teste:** notícias publicadas após **31/12/2019**, isto é, a partir de **01/01/2020**.
- **Avaliação online:** os dados pós-corte são processados em ordem cronológica, com previsão antes da atualização do modelo.


In [1]:
from pathlib import Path
import pandas as pd


In [2]:
CAMINHO_BASE = DATABASE_DIR / "Fake.br" / "corpus" / "full_texts"

COLUNAS_METADADOS = [
    "autor",
    "link",
    "categoria",
    "data_publicacao",
    "numero_tokens",
    "numero_palavras_sem_pontuacao",
    "numero_types",
    "numero_links",
    "numero_maiusculas",
    "numero_verbos",
    "numero_verbos_subjuntivo_imperativo",
    "numero_substantivos",
    "numero_adjetivos",
    "numero_adverbios",
    "numero_verbos_modais",
    "numero_pronomes_singular_primeira_segunda_pessoa",
    "numero_pronomes_plural_primeira_pessoa",
    "numero_pronomes",
    "pausalidade",
    "numero_caracteres",
    "tamanho_medio_sentenca",
    "tamanho_medio_palavra",
    "percentual_erros_ortograficos",
    "emotividade",
    "diversidade"
]

COLUNAS_NUMERICAS = COLUNAS_METADADOS[4:]


def ler_arquivo(caminho):
    try:
        return caminho.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return caminho.read_text(encoding="latin-1")

In [3]:
def carregar_classe(caminho_base, classe):

    pasta_textos = caminho_base / classe
    pasta_metadados = caminho_base / f"{classe}-meta-information"

    registros = []
    problemas = []

    arquivos_texto = sorted(
        pasta_textos.glob("*.txt"),
        key=lambda arquivo: int(arquivo.stem)
    )

    for caminho_texto in arquivos_texto:
        identificador = int(caminho_texto.stem)

        caminho_metadados = (
            pasta_metadados / f"{identificador}-meta.txt"
        )

        if not caminho_metadados.exists():
            problemas.append(
                f"Metadados não encontrados para o arquivo {identificador}"
            )
            continue

        texto_noticia = ler_arquivo(caminho_texto).strip()

        linhas_metadados = [
            linha.strip()
            for linha in ler_arquivo(caminho_metadados).splitlines()
        ]

        # Remove somente linhas vazias no final do arquivo.
        while linhas_metadados and linhas_metadados[-1] == "":
            linhas_metadados.pop()

        if len(linhas_metadados) != len(COLUNAS_METADADOS):
            problemas.append(
                f"Arquivo {caminho_metadados.name}: "
                f"esperadas {len(COLUNAS_METADADOS)} linhas, "
                f"encontradas {len(linhas_metadados)}"
            )
            continue

        metadados = dict(
            zip(COLUNAS_METADADOS, linhas_metadados)
        )

        registro = {
            "id": identificador,
            "texto": texto_noticia,
            "classe": classe,

            # Convenção adotada:
            # fake = 1
            # true = 0
            "rotulo": 1 if classe == "fake" else 0,

            **metadados
        }

        registros.append(registro)

    if problemas:
        print(f"Foram encontrados {len(problemas)} problemas.")

        for problema in problemas[:10]:
            print(problema)

    return pd.DataFrame(registros)




In [4]:
df_fake = carregar_classe(CAMINHO_BASE, "fake")
df_true = carregar_classe(CAMINHO_BASE, "true")

df = pd.concat(
    [df_fake, df_true],
    ignore_index=True
)

In [5]:
# Preserva o valor original para facilitar o diagnóstico de datas inválidas.
df["data_publicacao_original"] = df["data_publicacao"].copy()

# Converte as datas no formato dia/mês/ano.
df["data_publicacao"] = pd.to_datetime(
    df["data_publicacao_original"].astype("string").str.strip(),
    format="%d/%m/%Y",
    errors="coerce"
)

# Converte os atributos numéricos dos metadados.
df[COLUNAS_NUMERICAS] = df[COLUNAS_NUMERICAS].apply(
    pd.to_numeric,
    errors="coerce"
)


In [6]:
print(df.shape)
print(df.columns)


(7200, 30)
Index(['id', 'texto', 'classe', 'rotulo', 'autor', 'link', 'categoria',
       'data_publicacao', 'numero_tokens', 'numero_palavras_sem_pontuacao',
       'numero_types', 'numero_links', 'numero_maiusculas', 'numero_verbos',
       'numero_verbos_subjuntivo_imperativo', 'numero_substantivos',
       'numero_adjetivos', 'numero_adverbios', 'numero_verbos_modais',
       'numero_pronomes_singular_primeira_segunda_pessoa',
       'numero_pronomes_plural_primeira_pessoa', 'numero_pronomes',
       'pausalidade', 'numero_caracteres', 'tamanho_medio_sentenca',
       'tamanho_medio_palavra', 'percentual_erros_ortograficos', 'emotividade',
       'diversidade', 'data_publicacao_original'],
      dtype='str')


In [7]:
df.head()

,id,texto,classe,rotulo,autor,link,categoria,data_publicacao,numero_tokens,numero_palavras_sem_pontuacao,...,numero_pronomes_plural_primeira_pessoa,numero_pronomes,pausalidade,numero_caracteres,tamanho_medio_sentenca,tamanho_medio_palavra,percentual_erros_ortograficos,emotividade,diversidade,data_publicacao_original
0,1,Kátia Abreu diz que vai colocar sua expulsão e...,fake,1,mrk,https://ceticismopolitico.com/2017/11/30/katia...,politica,NaT,211,185,...,0,26,2.00000,815,14.2308,4.40541,0.000000,0.263158,0.648649,2017-11-30
1,2,Blog esquerdista dá a entender que reclamar de...,fake,1,None,https://ceticismopolitico.com/2017/11/28/blog-...,sociedade_cotidiano,NaT,356,300,...,0,30,3.29412,1321,17.6471,4.40333,0.013333,0.277372,0.623333,2017-11-28
2,3,"Alckmin diz que por ele PSDB desembarca, mas...",fake,1,None,https://ceticismopolitico.com/2017/11/28/alckm...,politica,NaT,274,224,...,0,14,3.57143,1075,16.0000,4.79911,0.000000,0.262136,0.669643,2017-11-28
3,4,Cara de pau não tem limites: Zé Celso aciona M...,fake,1,None,https://ceticismopolitico.com/2017/11/28/cara-...,politica,NaT,319,291,...,0,23,2.33333,1341,24.2500,4.60825,0.003436,0.182540,0.635739,2017-11-28
4,5,Temer resolve o problema de Luislinda: liberd...,fake,1,None,https://ceticismopolitico.com/2017/11/27/temer...,politica,NaT,162,133,...,0,12,2.63636,654,12.0909,4.91729,0.000000,0.303571,0.721804,2017-11-27


In [8]:
df["classe"].value_counts()

classe
fake    3600
true    3600
Name: count, dtype: int64

In [9]:
# ============================================================
# VALIDAÇÃO E COBERTURA TEMPORAL DA BASE
# ============================================================

print("Tipo da coluna:", df["data_publicacao"].dtype)
print("Datas inválidas ou ausentes:", df["data_publicacao"].isna().sum())
print("Data mais antiga:", df["data_publicacao"].min())
print("Data mais recente:", df["data_publicacao"].max())

print("\nQuantidade de notícias por ano:")
quantidade_por_ano = (
    df.dropna(subset=["data_publicacao"])
      .groupby(df["data_publicacao"].dt.year)
      .size()
      .sort_index()
)
print(quantidade_por_ano)

# Exibe exemplos de datas que não puderam ser convertidas, caso existam.
datas_invalidas = df.loc[
    df["data_publicacao"].isna(),
    ["id", "classe", "data_publicacao_original"]
]

if not datas_invalidas.empty:
    print("\nExemplos de datas inválidas ou ausentes:")
    display(datas_invalidas.head(20))


Tipo da coluna: datetime64[us]
Datas inválidas ou ausentes: 3531
Data mais antiga: 0201-01-06 00:00:00
Data mais recente: 2018-03-25 00:00:00

Quantidade de notícias por ano:
data_publicacao
201.0        3
2012.0       1
2013.0       1
2014.0       2
2015.0     237
2016.0    1585
2017.0    1545
2018.0     295
dtype: int64

Exemplos de datas inválidas ou ausentes:


,id,classe,data_publicacao_original
0,1,fake,2017-11-30
1,2,fake,2017-11-28
2,3,fake,2017-11-28
3,4,fake,2017-11-28
4,5,fake,2017-11-27
5,6,fake,2017-11-27
6,7,fake,2017-11-27
7,8,fake,2017-11-25
8,9,fake,2017-11-25
9,10,fake,2017-11-24


In [10]:
# ============================================================
# SALVAMENTO OPCIONAL DO DATAFRAME PROCESSADO
# ============================================================

# Pickle preserva os tipos das colunas, inclusive datetime.
df.to_pickle(str(PROCESSED_DIR / "fakebr_dataframe.pkl"))

# CSV facilita a inspeção em outros programas.
df.to_csv(
    str(PROCESSED_DIR / "fakebr_dataframe.csv"),
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d"
)


In [11]:
import numpy as np

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier


# ============================================================
# 1. BASE INICIAL: TODA A FAKE.BR
# ============================================================

# Remove registros sem texto ou rótulo
dados_iniciais = df.dropna(
    subset=["texto", "rotulo"]
).copy()

# Garante que os rótulos sejam números inteiros
dados_iniciais["rotulo"] = dados_iniciais["rotulo"].astype(int)


# ============================================================
# 2. VETORIZADOR PARA APRENDIZADO ONLINE
# ============================================================

# O HashingVectorizer não precisa aprender um vocabulário fixo.
# Por isso, ele pode transformar continuamente novos textos.
vetorizador = HashingVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    n_features=2**18,
    alternate_sign=False,
    norm="l2"
)

# Transforma toda a Fake.br em vetores numéricos
X_inicial = vetorizador.transform(
    dados_iniciais["texto"]
)

y_inicial = dados_iniciais["rotulo"].to_numpy()




In [12]:
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble  import RandomForestClassifier
from sklearn.svm import LinearSVC

# Crie agora para os modelos Perceptron, PassiveAggressiveClassifier, MultinomialNB e SGDClassifier, a partir daqui
#from sklearn.linear_model import Perceptron, PassiveAggressiveClassifier
###from sklearn.naive_bayes import MultinomialNB



cvC = 10
classifier = LogisticRegression()
pred = cross_val_predict(classifier, X_inicial, y_inicial, cv=cvC, verbose=True, n_jobs=2)

classifier2 = SGDClassifier()
pred2 = cross_val_predict(classifier2, X_inicial, y_inicial, cv=cvC, verbose=True, n_jobs=2)

classifier3 = BernoulliNB()
pred3 = cross_val_predict(classifier3, X_inicial, y_inicial, cv=cvC, verbose=True, n_jobs=2)

classifier4 = RandomForestClassifier()
pred4 = cross_val_predict(classifier4, X_inicial, y_inicial, cv=cvC, verbose=True, n_jobs=2)

classifier5 = LinearSVC()
pred5 = cross_val_predict(classifier5, X_inicial, y_inicial, cv=cvC, verbose=True, n_jobs=2)

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:   12.2s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    1.1s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.5s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:  1.2min finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    1.7s finished


In [13]:
from sklearn.metrics import accuracy_score, f1_score,precision_score,recall_score

score = accuracy_score(y_inicial, pred)
score2 = accuracy_score(y_inicial, pred2)
score3 = accuracy_score(y_inicial, pred3)
score4 = accuracy_score(y_inicial, pred4)
score5 = accuracy_score(y_inicial, pred5)

In [14]:
ps1 = precision_score(y_inicial, pred, average = 'macro')
r1  = recall_score(y_inicial, pred, average = 'macro')

ps2 = precision_score(y_inicial, pred2, average = 'macro')
r2 = recall_score(y_inicial, pred2, average = 'macro')

ps3 = precision_score(y_inicial, pred3, average = 'macro')
r3  = recall_score(y_inicial, pred3, average = 'macro')

ps4 = precision_score(y_inicial, pred4, average = 'macro')
r4  = recall_score(y_inicial, pred4, average = 'macro')

ps5 = precision_score(y_inicial, pred5, average = 'macro')
r5  = recall_score(y_inicial, pred5, average = 'macro')

In [15]:
y_train2 = y_inicial
X_train = X_inicial

In [16]:
f1_geral = f1_score(y_train2, pred, average = 'macro')
f1_geral2 = f1_score(y_train2, pred2, average = 'macro')
f1_geral3 = f1_score(y_train2, pred3, average = 'macro')
f1_geral4 = f1_score(y_train2, pred4, average = 'macro')
f1_geral5 = f1_score(y_train2, pred5, average = 'macro')
df1_geral = f1_score(y_train2, pred, average = None)
df1_gera2 = f1_score(y_train2, pred2, average = None)
df1_gera3 = f1_score(y_train2, pred3, average = None)
df1_gera4 = f1_score(y_train2, pred4, average = None)
df1_gera5 = f1_score(y_train2, pred5, average = None)

In [17]:
print("LogisticRegression - Accuracy:", score,", F1_Score:", f1_geral," Precision :",ps1," Recall :",r1,  "\n F1-Classe",df1_geral)
print("SGDClassifier - Accuracy:", score2,", F1_Score:", f1_geral2," Precision :",ps2," Recall :",r2,  "\n F1-Classe",df1_gera2)
print("ComplementNB - Accuracy:", score3,", F1_Score:", f1_geral3," Precision :",ps3," Recall :",r3,  "\n F1-Classe",df1_gera3)
print("RandomForestClassifier - Accuracy:", score4,", F1_Score:",f1_geral4," Precision :",ps4," Recall :",r4,  "\n F1-Classe",df1_gera4)
print("SVM - Accuracy:", score5,", F1_Score:", f1_geral5," Precision :",ps5," Recall :",r5,  "\n F1-Classe",df1_gera5)

LogisticRegression - Accuracy: 0.9518055555555556 , F1_Score: 0.951805547188463  Precision : 0.9518058693096314  Recall : 0.9518055555555556 
 F1-Classe [0.95178547 0.95182563]
SGDClassifier - Accuracy: 0.9593055555555555 , F1_Score: 0.9593038214147667  Precision : 0.9593838563995978  Recall : 0.9593055555555555 
 F1-Classe [0.95956948 0.95903817]
ComplementNB - Accuracy: 0.9116666666666666 , F1_Score: 0.9111963085962408  Precision : 0.9205771915861858  Recall : 0.9116666666666666 
 F1-Classe [0.90473337 0.91765924]
RandomForestClassifier - Accuracy: 0.955 , F1_Score: 0.9549902444133146  Precision : 0.9553948160612086  Recall : 0.9550000000000001 
 F1-Classe [0.9543276  0.95565289]
SVM - Accuracy: 0.9616666666666667 , F1_Score: 0.9616657083093744  Precision : 0.9617128379504618  Recall : 0.9616666666666667 
 F1-Classe [0.96185738 0.96147404]


In [18]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)

X_res, y_res = sm.fit_resample(X_train, y_train2)

X_train = X_res
y_train2 = y_res

from sklearn.model_selection import cross_val_predict
cvC = 10
classifier = LogisticRegression()
pred = cross_val_predict(classifier, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier2 = SGDClassifier()
pred2 = cross_val_predict(classifier2, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier3 = BernoulliNB()
pred3 = cross_val_predict(classifier3, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier4 = RandomForestClassifier()
pred4 = cross_val_predict(classifier4, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)

classifier5 = LinearSVC()
pred5 = cross_val_predict(classifier5, X_train, y_train2, cv=cvC, verbose=True, n_jobs=2)



[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:   11.0s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    1.0s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.4s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:  1.2min finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    1.7s finished


In [19]:
from sklearn.metrics import  f1_score, precision_score, recall_score
acc1 = accuracy_score(y_train2, pred)
acc2 = accuracy_score(y_train2, pred2)
acc3 = accuracy_score(y_train2, pred3)
acc4 = accuracy_score(y_train2, pred4)
acc5 = accuracy_score(y_train2, pred5)

ps1 = precision_score(y_train2, pred, average = 'macro')
r1  = recall_score(y_train2, pred, average = 'macro')

ps2 = precision_score(y_train2, pred2, average = 'macro')
r2 = recall_score(y_train2, pred2, average = 'macro')

ps3 = precision_score(y_train2, pred3, average = 'macro')
r3  = recall_score(y_train2, pred3, average = 'macro')

ps4 = precision_score(y_train2, pred4, average = 'macro')
r4  = recall_score(y_train2, pred4, average = 'macro')

ps5 = precision_score(y_train2, pred5, average = 'macro')
r5  = recall_score(y_train2, pred5, average = 'macro')



f1_geral = f1_score(y_train2, pred, average = 'macro')
f1_gera2 = f1_score(y_train2, pred2, average = 'macro')
f1_gera3 = f1_score(y_train2, pred3, average = 'macro')
f1_gera4 = f1_score(y_train2, pred4, average = 'macro')
f1_gera5 = f1_score(y_train2, pred5, average = 'macro')



df1_geral = f1_score(y_train2, pred, average = None)
df1_gera2 = f1_score(y_train2, pred2, average = None)
df1_gera3 = f1_score(y_train2, pred3, average = None)
df1_gera4 = f1_score(y_train2, pred4, average = None)
df1_gera5 = f1_score(y_train2, pred5, average = None)

In [20]:
print("LogisticRegression - Accuracy:", score,", F1_Score:", f1_geral," Precision :",ps1," Recall :",r1,  "\n F1-Classe",df1_geral)
print("SGDClassifier - Accuracy:", score2,", F1_Score:", f1_geral2," Precision :",ps2," Recall :",r2,  "\n F1-Classe",df1_gera2)
print("ComplementNB - Accuracy:", score3,", F1_Score:", f1_geral3," Precision :",ps3," Recall :",r3,  "\n F1-Classe",df1_gera2)
print("RandomForestClassifier - Accuracy:", score4,", F1_Score:"," Precision :",ps4," Recall :",r4,  "\n F1-Classe",df1_gera2)
print("SVM - Accuracy:", score5,", F1_Score:", f1_geral5," Precision :",ps5," Recall :",r5,  "\n F1-Classe",df1_gera2)

LogisticRegression - Accuracy: 0.9518055555555556 , F1_Score: 0.951805547188463  Precision : 0.9518058693096314  Recall : 0.9518055555555556 
 F1-Classe [0.95178547 0.95182563]
SGDClassifier - Accuracy: 0.9593055555555555 , F1_Score: 0.9593038214147667  Precision : 0.9581239913616726  Recall : 0.9580555555555555 
 F1-Classe [0.95831033 0.95779765]
ComplementNB - Accuracy: 0.9116666666666666 , F1_Score: 0.9111963085962408  Precision : 0.9205771915861858  Recall : 0.9116666666666666 
 F1-Classe [0.95831033 0.95779765]
RandomForestClassifier - Accuracy: 0.955 , F1_Score:  Precision : 0.9539899965897465  Recall : 0.9536111111111112 
 F1-Classe [0.95831033 0.95779765]
SVM - Accuracy: 0.9616666666666667 , F1_Score: 0.9616657083093744  Precision : 0.9617128379504618  Recall : 0.9616666666666667 
 F1-Classe [0.95831033 0.95779765]


In [21]:
print("LogisticRegression - Accuracy:", acc1,", F1_Score:", f1_geral, "\n",df1_geral)
print("SGDClassifier - Accuracy:", acc2,", F1_Score:", f1_gera2, "\n",df1_gera2)
print("ComplementNB - Accuracy:", acc3,", F1_Score:", f1_gera3, "\n",df1_gera3)
print("RandomForestClassifier - Accuracy:", acc4,", F1_Score:", f1_gera4, "\n",df1_gera4)
print("SVM - Accuracy:", acc5,", F1_Score:", f1_gera5, "\n",df1_gera5)

LogisticRegression - Accuracy: 0.9518055555555556 , F1_Score: 0.951805547188463 
 [0.95178547 0.95182563]
SGDClassifier - Accuracy: 0.9580555555555555 , F1_Score: 0.9580539890532949 
 [0.95831033 0.95779765]
ComplementNB - Accuracy: 0.9116666666666666 , F1_Score: 0.9111963085962408 
 [0.90473337 0.91765924]
RandomForestClassifier - Accuracy: 0.9536111111111111 , F1_Score: 0.9536014304219028 
 [0.95293123 0.95427163]
SVM - Accuracy: 0.9616666666666667 , F1_Score: 0.9616657083093744 
 [0.96185738 0.96147404]


In [22]:
import re
import numpy as np
import pandas as pd

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. CARREGA A BASE FAKERECOGNA
# ============================================================

CAMINHO_FAKERECOGNA = str(DATABASE_DIR / "FakeRecogna" / "FakeRecogna.xlsx")

df_recogna = pd.read_excel(CAMINHO_FAKERECOGNA)


# ============================================================
# 2. TRATAMENTO DAS DATAS
# ============================================================

def converter_data_fakerecogna(valor):
    """
    Converte os diferentes formatos de data encontrados
    na base FakeRecogna.
    """

    if pd.isna(valor):
        return pd.NaT

    texto = str(valor).strip()

    # Formatos como:
    # 27/03/2020
    # 27/03/2020 18h25
    # Publicado em 20/05/21 04:30
    padrao_br = re.search(
        r"(\d{1,2})/(\d{1,2})/(\d{2,4})",
        texto
    )

    if padrao_br:
        dia, mes, ano = map(int, padrao_br.groups())

        if ano < 100:
            ano += 2000

        try:
            return pd.Timestamp(
                year=ano,
                month=mes,
                day=dia
            )
        except ValueError:
            return pd.NaT


    # Formatos como:
    # 2020-03-31
    # 2020/11/20
    padrao_ano_primeiro = re.search(
        r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})",
        texto
    )

    if padrao_ano_primeiro:
        ano, mes, dia = map(
            int,
            padrao_ano_primeiro.groups()
        )

        try:
            return pd.Timestamp(
                year=ano,
                month=mes,
                day=dia
            )
        except ValueError:
            return pd.NaT

    return pd.NaT


df_recogna["data_publicacao"] = (
    df_recogna["Data"]
    .apply(converter_data_fakerecogna)
)


# ============================================================
# 3. PREPARA O TEXTO
# ============================================================

# Junta título, subtítulo e notícia.
# Isso nos dá mais informação textual para o classificador.

df_recogna["texto_modelo"] = (
    df_recogna[
        ["Titulo", "Subtitulo", "Noticia"]
    ]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.strip()
)


# ============================================================
# 4. CORRIGE OS RÓTULOS
# ============================================================

# Fake.br:
# fake = 1
# true = 0
#
# FakeRecogna:
# fake = 0
# true = 1
#
# Portanto, precisamos inverter.

df_recogna["rotulo"] = (
    1 - pd.to_numeric(
        df_recogna["Classe"],
        errors="coerce"
    )
)


df_recogna = df_recogna.dropna(
    subset=[
        "data_publicacao",
        "texto_modelo",
        "rotulo"
    ]
).copy()

df_recogna["rotulo"] = (
    df_recogna["rotulo"].astype(int)
)


# ============================================================
# 5. MODELO INICIAL
# ============================================================

modelo_online = SGDClassifier(
    random_state=42
)

# Treina inicialmente com a Fake.br
modelo_online.fit(
    X_inicial,
    y_inicial
)


# ============================================================
# 6. FUNÇÃO AUXILIAR PARA ESCOLHER O MÊS
# ============================================================

INICIO_POS_CORTE = pd.Timestamp("2020-01-01")


def selecionar_mes_fakerecogna(mes_apos_corte):

    if mes_apos_corte < 1:
        raise ValueError(
            "mes_apos_corte deve ser >= 1"
        )

    inicio = (
        INICIO_POS_CORTE
        + pd.DateOffset(
            months=mes_apos_corte - 1
        )
    )

    fim = (
        inicio
        + pd.DateOffset(months=1)
    )

    dados_mes = df_recogna[
        (df_recogna["data_publicacao"] >= inicio)
        &
        (df_recogna["data_publicacao"] < fim)
    ].copy()

    return dados_mes, inicio, fim


# ============================================================
# 7. FUNÇÃO DE TESTE
# ============================================================

def testar_mes_fakerecogna(mes_apos_corte):

    dados_mes, inicio, fim = (
        selecionar_mes_fakerecogna(
            mes_apos_corte
        )
    )

    if dados_mes.empty:
        print(
            f"Não existem notícias para "
            f"{inicio.strftime('%m/%Y')}."
        )
        return None

    X_teste = vetorizador.transform(
        dados_mes["texto_modelo"]
    )

    y_teste = (
        dados_mes["rotulo"]
        .to_numpy()
    )

    predicoes = modelo_online.predict(
        X_teste
    )


    accuracy = accuracy_score(
        y_teste,
        predicoes
    )

    f1 = f1_score(
        y_teste,
        predicoes,
        average="macro"
    )

    precision = precision_score(
        y_teste,
        predicoes,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_teste,
        predicoes,
        average="macro",
        zero_division=0
    )


    print(
        "========================================"
    )

    print(
        f"TESTE — {inicio.strftime('%m/%Y')}"
    )

    print(
        "========================================"
    )

    print(
        f"Número de notícias: {len(dados_mes)}"
    )

    print(
        f"Fake: {(y_teste == 1).sum()}"
    )

    print(
        f"True: {(y_teste == 0).sum()}"
    )

    print()

    print(
        f"Accuracy:  {accuracy:.4f}"
    )

    print(
        f"F1 Macro:  {f1:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall:    {recall:.4f}"
    )

    print()

    print(
        classification_report(
            y_teste,
            predicoes,
            labels=[0, 1],
            target_names=[
                "True",
                "Fake"
            ],
            zero_division=0
        )
    )

    print(
        "Matriz de confusão:"
    )

    print(
        confusion_matrix(
            y_teste,
            predicoes,
            labels=[0, 1]
        )
    )


    resultado = dados_mes[
        [
            "Titulo",
            "data_publicacao",
            "rotulo"
        ]
    ].copy()

    resultado["predicao"] = predicoes

    return resultado

In [23]:
resultado_mes_1 = testar_mes_fakerecogna(1)

TESTE — 01/2020
Número de notícias: 175
Fake: 97
True: 78

Accuracy:  0.5714
F1 Macro:  0.3976
Precision: 0.7820
Recall:    0.5192

              precision    recall  f1-score   support

        True       1.00      0.04      0.07        78
        Fake       0.56      1.00      0.72        97

    accuracy                           0.57       175
   macro avg       0.78      0.52      0.40       175
weighted avg       0.76      0.57      0.43       175

Matriz de confusão:
[[ 3 75]
 [ 0 97]]


In [24]:
resultado_mes_2 = testar_mes_fakerecogna(2)

TESTE — 02/2020
Número de notícias: 208
Fake: 108
True: 100

Accuracy:  0.5240
F1 Macro:  0.3528
Precision: 0.7609
Recall:    0.5050

              precision    recall  f1-score   support

        True       1.00      0.01      0.02       100
        Fake       0.52      1.00      0.69       108

    accuracy                           0.52       208
   macro avg       0.76      0.51      0.35       208
weighted avg       0.75      0.52      0.37       208

Matriz de confusão:
[[  1  99]
 [  0 108]]


In [25]:
resultado_mes_3 = testar_mes_fakerecogna(3)

TESTE — 03/2020
Número de notícias: 581
Fake: 224
True: 357

Accuracy:  0.3873
F1 Macro:  0.2814
Precision: 0.6931
Recall:    0.5014

              precision    recall  f1-score   support

        True       1.00      0.00      0.01       357
        Fake       0.39      1.00      0.56       224

    accuracy                           0.39       581
   macro avg       0.69      0.50      0.28       581
weighted avg       0.76      0.39      0.22       581

Matriz de confusão:
[[  1 356]
 [  0 224]]


In [26]:
def treinar_mes_fakerecogna(mes_apos_corte):

    dados_mes, inicio, fim = (
        selecionar_mes_fakerecogna(
            mes_apos_corte
        )
    )

    if dados_mes.empty:
        print(
            f"Não existem notícias para "
            f"{inicio.strftime('%m/%Y')}."
        )
        return modelo_online


    X_treino_mes = vetorizador.transform(
        dados_mes["texto_modelo"]
    )

    y_treino_mes = (
        dados_mes["rotulo"]
        .to_numpy()
    )


    # Atualiza o modelo sem apagar
    # o conhecimento adquirido anteriormente.
    modelo_online.partial_fit(
        X_treino_mes,
        y_treino_mes,
        classes=np.array([0, 1])
    )


    print(
        "========================================"
    )

    print(
        f"TREINAMENTO — "
        f"{inicio.strftime('%m/%Y')}"
    )

    print(
        "========================================"
    )

    print(
        f"Notícias utilizadas: "
        f"{len(dados_mes)}"
    )

    print(
        f"Fake: "
        f"{(y_treino_mes == 1).sum()}"
    )

    print(
        f"True: "
        f"{(y_treino_mes == 0).sum()}"
    )

    print(
        "\nModelo atualizado com sucesso."
    )

    return modelo_online

In [27]:
teste1 = testar_mes_fakerecogna(1)

TESTE — 01/2020
Número de notícias: 175
Fake: 97
True: 78

Accuracy:  0.5714
F1 Macro:  0.3976
Precision: 0.7820
Recall:    0.5192

              precision    recall  f1-score   support

        True       1.00      0.04      0.07        78
        Fake       0.56      1.00      0.72        97

    accuracy                           0.57       175
   macro avg       0.78      0.52      0.40       175
weighted avg       0.76      0.57      0.43       175

Matriz de confusão:
[[ 3 75]
 [ 0 97]]


In [28]:
teste2 = testar_mes_fakerecogna(2)

TESTE — 02/2020
Número de notícias: 208
Fake: 108
True: 100

Accuracy:  0.5240
F1 Macro:  0.3528
Precision: 0.7609
Recall:    0.5050

              precision    recall  f1-score   support

        True       1.00      0.01      0.02       100
        Fake       0.52      1.00      0.69       108

    accuracy                           0.52       208
   macro avg       0.76      0.51      0.35       208
weighted avg       0.75      0.52      0.37       208

Matriz de confusão:
[[  1  99]
 [  0 108]]


In [29]:
teste3 = testar_mes_fakerecogna(3)

TESTE — 03/2020
Número de notícias: 581
Fake: 224
True: 357

Accuracy:  0.3873
F1 Macro:  0.2814
Precision: 0.6931
Recall:    0.5014

              precision    recall  f1-score   support

        True       1.00      0.00      0.01       357
        Fake       0.39      1.00      0.56       224

    accuracy                           0.39       581
   macro avg       0.69      0.50      0.28       581
weighted avg       0.76      0.39      0.22       581

Matriz de confusão:
[[  1 356]
 [  0 224]]


In [30]:
resultado_mes_1_postreinamento = testar_mes_fakerecogna(1)

TESTE — 01/2020
Número de notícias: 175
Fake: 97
True: 78

Accuracy:  0.5714
F1 Macro:  0.3976
Precision: 0.7820
Recall:    0.5192

              precision    recall  f1-score   support

        True       1.00      0.04      0.07        78
        Fake       0.56      1.00      0.72        97

    accuracy                           0.57       175
   macro avg       0.78      0.52      0.40       175
weighted avg       0.76      0.57      0.43       175

Matriz de confusão:
[[ 3 75]
 [ 0 97]]


In [31]:
mport numpy as np
import pandas as pd
import sklearn as skl
import scipy
import datetime

import pickle

from sklearn import feature_extraction
from sklearn import linear_model
from sklearn import svm
from sklearn import neighbors
from sklearn import tree
from sklearn import ensemble
from sklearn import naive_bayes
from sklearn import model_selection
from sklearn import metrics

#from sklearn.externals import joblib
#import sklearn.external.joblib as extjoblib
#import joblib

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pylab

import collections

import os
import sys

def plota_offline(methodsList, baseline = 0.90, annotateDist = 0.01, titulo = None):

    markers = ['o',"s",'^','v',"1","<",'*','X']
    colors = ['orange','brown','blue','green','purple','olive','brown']
    styles = ['-','-',':',':','-.','-.']

    for i in range( len(methodsList) ):

        metodo, res, baseline = methodsList[i]


        if baseline is not None:

            # verifica se e um numero
            if isinstance(baseline, (int, float)):
                plt.plot([0, 11],[baseline,baseline],color='red',linestyle='--', linewidth=1.0,zorder=1, label=metodo + ' (expectation)')#label='Nov and Dec 2019')
            else:
               plt.plot(np.arange(14), baseline, color=colors[i], linestyle=':', marker=markers[i], linewidth=2.0, label=None , markersize=6)#label='Nov and Dec 2019')

               #value2anotate = baseline[0]
               #plt.annotate('%1.2f' %np.round(value2anotate,2),xy=(0-annotateDist,value2anotate+annotateDist),fontsize='large',rotation=0)

               #value2anotate = baseline[1]
               #plt.annotate('%1.2f' %np.round(value2anotate,2),xy=(1-annotateDist,value2anotate+annotateDist),fontsize='large',rotation=0)



        #df_count[['F-measure']].plot(marker='o')]
        plt.plot(np.arange(14), res[ res.columns[1] ], marker=markers[i], label=res.columns[1], linestyle='-', color = colors[i], linewidth=1.0, markersize=6)

            # escreve o valor da medida no grafico
            #for j in range( len(res) ):
            #
            #    if i == 0 or i == 1:
            #        plt.annotate('%1.2f' %np.round(res[ res.columns[i+1] ].iloc[j],2),xy=(j-annotateDist,res[ res.columns[i+1] ].iloc[j]+annotateDist),fontsize='large',rotation=0)


        plt.plot([1, 2],[baseline[1],res[metodo].iloc[2]],color='red',linestyle=':', linewidth=1.0,zorder=1, label=None)#label='Nov and Dec 2019')


    ymin, ymax = plt.ylim()   # return the current xlim
    xmin, xmax = plt.xlim()   # return the current xlim

    ymax = 0.9
    ymin = 0.10
    plt.ylim([ymin,ymax]) # set the xlim

    plt.annotate('Expectation at the end of 2019',xy=(0.6, ymax-0.08), xytext=(1.5, ymax-0.05),arrowprops=dict(facecolor='black', shrink=0.05),fontsize='large',rotation=0,bbox=dict(boxstyle="round", fc="w"))
    plt.annotate('}',xy=(-0.2, ymax-0.1), xytext=(-0.2, ymax-0.1),fontsize=50,rotation=90)

    plt.yticks(np.arange(np.round(ymin,2), np.round(ymax,2), 0.02))

    #plt.axvline(x=1, color='gray',linestyle='--',linewidth=1.0)
    # adiciona uma area colorida
    plt.axvspan(0, 1, alpha=0.2, color='gray')
    #plt.axhspan(0, 1, 1, 2, alpha=0.2, color='gray')
    #plt.fill_between(plt.ylim(), plt.ylim(), 1)
    #plt.margins(x=0)


    plt.ylabel(r'F-measure',fontsize='large')
    #plt.xlabel(r'Month',fontsize='large')

    months = ['Nov 2019','Dec 2019','Jan 2020', 'Feb 2020', 'Mar 2020', 'Apr 2020', 'May 2020', 'Jun 2020', 'Jul 2020', 'Aug 2020', 'Sep 2020', 'Oct 2020', 'Nov 2020', 'Dec 2020']
    #plt.xticks( np.arange(0,12), np.arange(1,13) )
    plt.xticks( np.arange(0,14), months)
    locs, labels = plt.xticks()
    plt.setp(labels, rotation=45, ha='right',fontsize='large') #rotate x labels #ha is the horizontal alignment of the labels

    #plt.setp( fontsize='large' ) #rotate x labels #ha is the horizontal alignment of the labels
    plt.yticks(fontsize='large')

    #plt.legend(fontsize='large')
    #plt.legend(bbox_to_anchor=(0., 1.005, 1., .102), loc=3, ncol=4, mode="expand", borderaxespad=0., fontsize='large')
    plt.legend(bbox_to_anchor=(0., 1.005, 1., .102), loc=3, ncol=4, mode="expand", borderaxespad=0., fontsize='large')
    #plt.legend(bbox_to_anchor=(1.01, 1), loc=2, borderaxespad=0.)
    #plt.legend(loc='best', fontsize='large', ncol=4)

    # cria um título para o gráfico
    # if titulo is not None:
    #     #plt.title(titulo, fontsize='x-large', pad=70)
    #     plt.title(titulo, fontsize='x-large')

    #     # altura do titulo
    #     #plt.rcParams['axes.titley'] = 1.20    # y is in axes-relative coordinates.
    #     plt.rcParams['axes.titlepad'] = -60  # pad is in points...

    plt.grid(axis='y')

    #plt.rcParams['xtick.major.pad']='1'

    # salva em uma figura png
    #titulo = titulo.replace(' ','')
    #titulo = titulo.replace(')','')
    #titulo = titulo.replace('(','')
    #plt.savefig(path, format='png', bbox_inches='tight', dpi=500)

    #plt.show()



def plota_online(methodsList, baseline = 0.90, annotateDist = 0.01, titulo = None):

    #df_count[['countClass0','countClass2']].plot(marker='.', linestyle='-',)

    #cria um gráfico de barras com a frequência de cada classe
    #sns.barplot(x="date", hue=['countClass0'], data=df_count)

    markers = ['o',"s",'^','v',"1","<",'*','X']
    colors = ['orange','brown','blue','green','purple','olive','brown']
    styles = ['-','-',':',':','-.','-.']

    for i in range( len(methodsList) ):

        metodo, res, baseline = methodsList[i]


        if baseline is not None:

            # verifica se e um numero
            if isinstance(baseline, (int, float)):
                plt.plot([0, 11],[baseline,baseline],color='red',linestyle='--', linewidth=1.0,zorder=1, label=metodo + ' (expectation)')#label='Nov and Dec 2019')
            else:
               plt.plot(np.arange(14), baseline, color=colors[i], linestyle=':', marker=markers[i], linewidth=2.0, label=metodo + ' (expectation)' , markersize=6)#label='Nov and Dec 2019')

               #value2anotate = baseline[0]
               #plt.annotate('%1.2f' %np.round(value2anotate,2),xy=(0-annotateDist,value2anotate+annotateDist),fontsize='large',rotation=0)

               #value2anotate = baseline[1]
               #plt.annotate('%1.2f' %np.round(value2anotate,2),xy=(1-annotateDist,value2anotate+annotateDist),fontsize='large',rotation=0)



        #df_count[['F-measure']].plot(marker='o')]
        plt.plot(np.arange(14), res[ res.columns[2] ], marker=markers[i], label=res.columns[2], linestyle='-', color = colors[i], linewidth=1.0, markersize=6)

            # escreve o valor da medida no grafico
            #for j in range( len(res) ):
            #
            #    if i == 0 or i == 1:
            #        plt.annotate('%1.2f' %np.round(res[ res.columns[i+1] ].iloc[j],2),xy=(j-annotateDist,res[ res.columns[i+1] ].iloc[j]+annotateDist),fontsize='large',rotation=0)


        #plt.plot([1, 2],[baseline[1],res[metodo + ' (offline)'].iloc[2]],color='red',linestyle='--', linewidth=1.0,zorder=1, label=None)#label='Nov and Dec 2019')
        plt.plot([1, 2],[baseline[1],res[metodo + ' (online)'].iloc[2]],color='red',linestyle=':', linewidth=1.0,zorder=1, label=None)#label='Nov and Dec 2019')



    ymin, ymax = plt.ylim()   # return the current xlim
    xmin, xmax = plt.xlim()   # return the current xlim

    ymax = 0.9
    ymin = 0.10
    plt.ylim([ymin,ymax]) # set the xlim

    plt.annotate('Expectation at the end of 2019',xy=(0.6, ymax-0.08), xytext=(1.5, ymax-0.05),arrowprops=dict(facecolor='black', shrink=0.05),fontsize='large',rotation=0,bbox=dict(boxstyle="round", fc="w"))
    plt.annotate('}',xy=(-0.2, ymax-0.1), xytext=(-0.2, ymax-0.1),fontsize=50,rotation=90)

    plt.yticks(np.arange(np.round(ymin,2), np.round(ymax,2), 0.02))

    #plt.axvline(x=1, color='gray',linestyle='--',linewidth=1.0)
    # adiciona uma area colorida
    plt.axvspan(0, 1, alpha=0.2, color='gray')
    #plt.axhspan(0, 1, 1, 2, alpha=0.2, color='gray')
    #plt.fill_between(plt.ylim(), plt.ylim(), 1)
    #plt.margins(x=0)


    plt.ylabel(r'F-measure',fontsize='large')
    #plt.xlabel(r'Month',fontsize='large')

    months = ['Nov 2019','Dec 2019','Jan 2020', 'Feb 2020', 'Mar 2020', 'Apr 2020', 'May 2020', 'Jun 2020', 'Jul 2020', 'Aug 2020', 'Sep 2020', 'Oct 2020', 'Nov 2020', 'Dec 2020']
    #plt.xticks( np.arange(0,12), np.arange(1,13) )
    plt.xticks( np.arange(0,14), months)
    locs, labels = plt.xticks()
    plt.setp(labels, rotation=45, ha='right',fontsize='large') #rotate x labels #ha is the horizontal alignment of the labels

    #plt.setp( fontsize='large' ) #rotate x labels #ha is the horizontal alignment of the labels
    plt.yticks(fontsize='large')

    #plt.legend(fontsize='large')
    #plt.legend(bbox_to_anchor=(0., 1.005, 1., .102), loc=3, ncol=4, mode="expand", borderaxespad=0., fontsize='large')
    plt.legend(bbox_to_anchor=(0., 1.005, 1., .102), loc=3, ncol=3, mode="expand", borderaxespad=0., fontsize='large')
    #plt.legend(bbox_to_anchor=(1.01, 1), loc=2, borderaxespad=0.)
    #plt.legend(loc='best', fontsize='large', ncol=4)

    # cria um título para o gráfico
    # if titulo is not None:
    #     #plt.title(titulo, fontsize='x-large', pad=70)
    #     plt.title(titulo, fontsize='x-large')

    #     # altura do titulo
    #     #plt.rcParams['axes.titley'] = 1.20    # y is in axes-relative coordinates.
    #     plt.rcParams['axes.titlepad'] = -60  # pad is in points...

    plt.grid(axis='y')

    #plt.rcParams['xtick.major.pad']='1'

    # salva em uma figura png
    #titulo = titulo.replace(' ','')
    #titulo = titulo.replace(')','')
    #titulo = titulo.replace('(','')
    #plt.savefig(path, format='png', bbox_inches='tight', dpi=500)

    #plt.show()


def plota_online2(methodsList, baseline = 0.90, annotateDist = 0.01, titulo = None):

    #df_count[['countClass0','countClass2']].plot(marker='.', linestyle='-',)

    #cria um gráfico de barras com a frequência de cada classe
    #sns.barplot(x="date", hue=['countClass0'], data=df_count)

    markers = ['o',"s",'^','v',"1","<",'*','X']
    colors = ['orange','brown','blue','green','purple','olive','brown']
    styles = ['-','-',':',':','-.','-.']

    for i in range( len(methodsList) ):

        metodo, res, baseline = methodsList[i]


        if baseline is not None:

            # verifica se e um numero
            if isinstance(baseline, (int, float)):
                plt.plot([0, 11],[baseline,baseline],color='red',linestyle='--', marker=markers[i], linewidth=1.0, markersize=6, zorder=1, label=metodo + ' (expectation)')#label='Nov and Dec 2019')
            else:
               plt.plot(np.arange(14), baseline, color=colors[i], linestyle=':', marker=markers[i], linewidth=2.0, label=None , markersize=6)#label='Nov and Dec 2019')

               #value2anotate = baseline[0]
               #plt.annotate('%1.2f' %np.round(value2anotate,2),xy=(0-annotateDist,value2anotate+annotateDist),fontsize='large',rotation=0)

               #value2anotate = baseline[1]
               #plt.annotate('%1.2f' %np.round(value2anotate,2),xy=(1-annotateDist,value2anotate+annotateDist),fontsize='large',rotation=0)



        #df_count[['F-measure']].plot(marker='o')]
        plt.plot(np.arange(14), res[ res.columns[2] ], marker=markers[i], label=res.columns[2], linestyle='-', color = colors[i], linewidth=1.0, markersize=6)
        plt.plot(np.arange(14), res[ res.columns[1] ], marker=markers[i], label=res.columns[1], dashes=[3, 2,  2, 2, 2, 2], color = colors[i], linewidth=2.0, markersize=6)

            # escreve o valor da medida no grafico
            #for j in range( len(res) ):
            #
            #    if i == 0 or i == 1:
            #        plt.annotate('%1.2f' %np.round(res[ res.columns[i+1] ].iloc[j],2),xy=(j-annotateDist,res[ res.columns[i+1] ].iloc[j]+annotateDist),fontsize='large',rotation=0)


        plt.plot([1, 2],[baseline[1],res[metodo + ' (offline)'].iloc[2]],color=colors[i],linestyle=':', linewidth=1.0,zorder=1, label=None)#label='Nov and Dec 2019')
        plt.plot([1, 2],[baseline[1],res[metodo + ' (online)'].iloc[2]],color=colors[i],linestyle=':', linewidth=1.0,zorder=1, label=None)#label='Nov and Dec 2019')

    #plt.annotate('{',xy=(1, 0.80), xytext=(1, 0.82),arrowprops=dict(facecolor='black', shrink=0.05),fontsize='large',rotation=90)

    ymin, ymax = plt.ylim()   # return the current xlim
    xmin, xmax = plt.xlim()   # return the current xlim

    ymax = 0.9
    ymin = 0.10
    plt.ylim([ymin,ymax]) # set the xlim

    plt.annotate('Expectation at the end of 2019',xy=(0.6, ymax-0.08), xytext=(1.5, ymax-0.05),arrowprops=dict(facecolor='black', shrink=0.05),fontsize='large',rotation=0,bbox=dict(boxstyle="round", fc="w"))
    plt.annotate('}',xy=(-0.2, ymax-0.1), xytext=(-0.2, ymax-0.1),fontsize=50,rotation=90)

    plt.yticks(np.arange(np.round(ymin,2), np.round(ymax,2), 0.02))
    #

    #plt.axvline(x=1, color='gray',linestyle='--',linewidth=1.0)
    # adiciona uma area colorida
    plt.axvspan(0, 1, alpha=0.2, color='gray')
    #plt.axhspan(0, 1, 1, 2, alpha=0.2, color='gray')
    #plt.fill_between(plt.ylim(), plt.ylim(), 1)
    #plt.margins(x=0)


    plt.ylabel(r'F-measure',fontsize='large')
    #plt.xlabel(r'Month',fontsize='large')

    months = ['Nov 2019','Dec 2019','Jan 2020', 'Feb 2020', 'Mar 2020', 'Apr 2020', 'May 2020', 'Jun 2020', 'Jul 2020', 'Aug 2020', 'Sep 2020', 'Oct 2020', 'Nov 2020', 'Dec 2020']
    #plt.xticks( np.arange(0,12), np.arange(1,13) )
    plt.xticks( np.arange(0,14), months)
    locs, labels = plt.xticks()
    plt.setp(labels, rotation=45, ha='right',fontsize='large') #rotate x labels #ha is the horizontal alignment of the labels

    #plt.setp( fontsize='large' ) #rotate x labels #ha is the horizontal alignment of the labels
    plt.yticks(fontsize='large')

    #plt.legend(fontsize='large')
    #plt.legend(bbox_to_anchor=(0., 1.005, 1., .102), loc=3, ncol=4, mode="expand", borderaxespad=0., fontsize='large')
    plt.legend(bbox_to_anchor=(0., 1.005, 1., .102), loc=3, ncol=4, mode="expand", borderaxespad=0., fontsize='large')
    #plt.legend(bbox_to_anchor=(1.01, 1), loc=2, borderaxespad=0.)
    #plt.legend(loc='best', fontsize='large', ncol=4)

    # cria um título para o gráfico
    # if titulo is not None:
    #     #plt.title(titulo, fontsize='x-large', pad=70)
    #     plt.title(titulo, fontsize='x-large')

    #     # altura do titulo
    #     #plt.rcParams['axes.titley'] = 1.20    # y is in axes-relative coordinates.
    #     plt.rcParams['axes.titlepad'] = -60  # pad is in points...

    plt.grid(axis='y')

    #plt.rcParams['xtick.major.pad']='1'

    # salva em uma figura png
    #titulo = titulo.replace(' ','')
    #titulo = titulo.replace(')','')
    #titulo = titulo.replace('(','')
    #plt.savefig(path, format='png', bbox_inches='tight', dpi=500)

    #plt.show()

def return_dfMethod_toPlot( df_scores, metodo, isOnline = None, medidaPrincipal = None ):

    df_method = df_scores[ df_scores['metodo'] == metodo ]

    # transforma a tabela para que cada metodo fique em uma linha e cada base em uma coluna
    df_method = df_method.pivot(index='month', columns='base_dados', values=medidaPrincipal)
    df_method = df_method.reset_index()

    df_method = df_method.sort_values(['month'], ascending=True)

    # arruma os valores de 2019
    baseline = np.zeros(14)
    baseline[2:] = np.nan
    baseline[0] = df_method['2019 (offline)-TrainOct'].iloc[10]
    baseline[1] = df_method['2019 (offline)-TrainNov'].iloc[11]


    if isOnline:

        # arruma os valores de 2020
        res = {}
        res['month'] = list(np.arange(1,15))
        res[metodo + ' (offline)'] = np.concatenate([np.zeros(2)*np.nan,df_method['2020 (offline)-TrainDec'].values])
        res[metodo + ' (online)'] = np.concatenate([np.zeros(2)*np.nan,df_method['2020-TrainDec'].values])
        #res['Covid-19 (online)'] = np.concatenate([np.zeros(2)*np.nan,df_method['2020 (Covid-19)-TrainDec'].values])
        #res['Elections (online)'] = np.concatenate([np.zeros(2)*np.nan,df_method['2020 (elections)-TrainDec'].values])
        df_method = pd.DataFrame(res)

    else:

        # arruma os valores de 2020
        res = {}
        res['month'] = list(np.arange(1,15))
        res[metodo] = np.concatenate([np.zeros(2)*np.nan,df_method['2020 (offline)-TrainDec'].values])
        #res['Covid-19'] = np.concatenate([np.zeros(2)*np.nan,df_method['2020 (Covid-19) (offline)-TrainDec'].values])
        #res['Elections'] = np.concatenate([np.zeros(2)*np.nan,df_method['2020 (elections) (offline)-TrainDec'].values])
        df_method = pd.DataFrame(res)

    return df_method, baseline



if __name__ == "__main__":

    # Resultados informados pelo usuário (F-measure Macro)
    resultados = {
        'Perceptron': [
            0.5064102564102564, 0.39488592864798655, 0.30900896269257794,
            0.2792484015243485, 0.2902939714820903, 0.318907066959015,
            0.28595722351249186, 0.2846623563218391, 0.3254309562692796,
            0.3566808526594049, 0.37652550155745546, 0.2788889548341148,
            0.27125851269662554, 0.25255382681125255
        ],
        'SGDClassifier': [
            0.3976318325760705, 0.35275813295615277, 0.28140026126351486,
            0.2405786586917612, 0.2650077911016908, 0.27641379310344827,
            0.2543448744896174, 0.27010960818463403, 0.2751145379371997,
            0.3273521969174143, 0.3498037701859358, 0.25073288968713664,
            0.23217251391688704, 0.22300254093732355
        ],
        'PassiveAggressiveClassifier': [
            0.5064102564102564, 0.39488592864798655, 0.3149919519371739,
            0.2758226358445922, 0.2910862523843142, 0.3319672131147541,
            0.29146429967063026, 0.2875355719977234, 0.3175905606971345,
            0.35771154794617066, 0.3839758125472411, 0.27747844827586204,
            0.26581058335128505, 0.24965587159531374
        ],
        'MultinomialNB': [
            0.48423224285293254, 0.6135895152054969, 0.5258929131881238,
            0.5159410893734833, 0.5097352752438896, 0.49186463480631404,
            0.5345706831119544, 0.5307603496618836, 0.5273278393482997,
            0.44467127075822727, 0.3698477921175994, 0.49070945945945943,
            0.5430820642516209, 0.5081559505522639
        ]
    }

    pathFig = str(RESULT_DIR / 'resultados_4_modelos.png')
    annotateDist = 0.005

    # Mantém o mesmo formato esperado por plota_offline():
    # baseline = Nov/Dez 2019; res = Jan-Dez 2020.
    methodsList = []

    for metodo, valores in resultados.items():
        valores = np.asarray(valores, dtype=float)

        if len(valores) != 14:
            raise ValueError(f'{metodo}: esperado vetor com 14 resultados, recebido {len(valores)}.')

        baseline = np.full(14, np.nan)
        baseline[0] = valores[0]
        baseline[1] = valores[1]

        res = pd.DataFrame({
            'month': np.arange(1, 15),
            metodo: np.concatenate([np.array([np.nan, np.nan]), valores[2:]])
        })

        methodsList.append([metodo, res, baseline])

    plt.figure(figsize=(10, 8))
    plota_offline(methodsList, annotateDist=annotateDist)
    plt.savefig(pathFig, format='png', bbox_inches='tight', dpi=500)
    print(f'Figura salva em: {pathFig}')


SyntaxError: invalid syntax (489279833.py, line 1)

In [ ]:
# Exemplos de consulta isolada das métricas:
#
# SGDClassifier.numero_noticias(2)
# SGDClassifier.fake(2)
# SGDClassifier.true(2)
 SGDClassifier.accuracy(2)
 SGDClassifier.f1macro(2)
 SGDClassifier.precision(2)
 SGDClassifier.recall(2)
# SGDClassifier.matriz_confusao(2)

 Perceptron.recall(2)
 PassiveAggressiveClassifier.f1macro(2)
 MultinomialNB.accuracy(2)
